# Modelos Lineares Generalizados
## Motivação
Como vimos até aqui, o modelo linear é bastante flexível e poderoso. No entanto, o modelo normal tem uma limitação importante: o suporte da distribuição dos dados, que fica limitado a $\mathbb{R}$. Como muitos fenômenos de interesse podem ser modelados como variáveis aleatórias com suporte restrito (e.g. $(0,1)$ ou $\mathbb{R}+$) e também discreto, como é o caso do modelos de contagem.

A solução se encontra na formulação dos chamados modelos lineares generalizados (generalised linear models, GLM), em que o preditor linear é conectado à esperança condicional por meio de uma função especial, chamada **função de ligação**.

## Estrutura Básica de um GLM

Sejam $Y=(Y_1,...,Y_n)$ e $X$ o vetor de variáveis dependentes e a matriz $n \times P$ de desenho, respectivamente. Defina $\mu_i(X) = \mu_i = \mathbb{E}[Y_i|X]$ como média condicional de cada $Y_i$. Em um GLM, escrevemos
$$
    g(\mu) = X \beta
$$

onde $g(\cdot)$ é uma função monotônica e diferenciável, chamada de função de ligação. Além disso, suponha que cada $Y_i$ tenha distribuição da família exponencial com parâmetro canônico $\theta_i$, isto é
$$
    f_{Y_i}(y_i|\theta_i) = exp\left\{y_i \theta_i + b(\theta_i) + c(y_i) \right\}     \\

    \Rightarrow f(y|\theta) = \prod_{i=1}^n f_{Y_i}(y_i|\theta_i) = exp\left\{\sum_{i=1}^n y_i \theta_i + \sum_{i=1}^n b(\theta_i) + \sum_{i=1}^n c(y_i) \right\}
$$

Suponha que g é a função de ligação canônica, isto é, que $g(\mu_i) = x_i^T \beta = \theta_i$. Então, a
log-verossimilhança para $\beta$ é
$$
    \mathcal{L}(\beta) = \sum_{i=1}^n y_i x_i^T \beta + \sum_{i=1}^n b(x_i^T \beta) + \sum_{i=1}^n c(y_i)
$$
ou, em notação matricial:
$$
    \mathcal{L}(\beta) = y^T X \beta - \mathbb{1}^T b(X \beta) + \mathbb{1}^T c(y)
$$
dessa forma, facilmente encontramos as derivadas com relação a $\beta$:
$$
\begin{align}
    \frac{\partial \mathcal{L}}{\partial \beta_k} (\beta) &= \sum_{i=1}^n y_i x_{ik} - \sum_{i=1}^n x_{ik} b'(x_i^T \beta)       \\
    \frac{\partial^2 \mathcal{L}}{\partial \beta_k \partial \beta_l} (\beta) &= - \sum_{i=1}^n x_{ik} x_{il} b''(x_i^T \beta)
\end{align}
$$
e em forma matricial:
$$
\begin{align}
    \nabla \mathcal{L}(\beta) &= X^T (y - b'(X \beta))       \\
    \nabla^2 \mathcal{L}(\beta) &= - X^T \text{diag}(b''(X\beta)) X
\end{align}
$$

## Regressão Poisson

Considere $Y_i|X \sim \text{Poisson}(\theta_i)$. Vamos expressar a fdp em termos da familia exponencial. Temos então:
$$
    f(y|\theta) = \frac{e^{-\theta} \theta^y}{y!} = \text{exp} \left\{ y \ln\theta - \theta - \ln(y!) \right\}
$$

então o parâmetro canônico é $\eta = \ln\theta$, $b(\eta) = e^\eta$ e $c(y)=-\ln y!$. Lembre-se que a função de ligação canônica é aquela que conecta o parâmetro canônico $\eta_i$ com $\mu_i$ e $x_i^T \beta$ de modo que $\eta_i = x_i^T \beta$. Logo, como $\mu_i = \text{exp}(\eta_i)$, temos que $g(t) = \ln(t)$ e, portanto, $\mu_i = \text{exp}(x_i^T \beta)$. Agora vamos obter a função score e a Hessiana, necessárias para a estimação dos parâmetros. Pelos cálculos anteriores, a função score é dada por

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd

In [6]:
def glm1(y, X, bp, bpp, tol=1e-6, max_iter=100):
    p = X.shape[1]
    beta_k = np.zeros(p)
    list_beta = []
    current_error = 1
    
    while current_error > tol:
        eta_k = X @ beta_k
        z_k = y - bp(eta_k)
        w_k = bpp(eta_k)
        
        X_k = (w_k**0.5)[:, None] * X
        wz_k = (w_k**-0.5) * z_k
        
        Q_k, R_k = np.linalg.qr(X_k)
        a_k = np.linalg.solve(R_k, Q_k.T @ wz_k)
        
        new_beta = a_k + beta_k
        current_error = np.max(np.abs(beta_k - new_beta))
        beta_k = new_beta
        list_beta.append(beta_k.copy())
        
        if len(list_beta) >= max_iter:
            break
    
    return np.array(list_beta)

In [7]:
def poi_reg(y, X):
    bp = lambda theta: np.exp(theta)
    bpp = lambda theta: np.exp(theta)
    return glm1(y, X, bp, bpp)

In [8]:
# Simula os dados
np.random.seed(20032025)

n = 500
X = np.column_stack([np.ones(n), np.random.normal(size=n), np.random.uniform(size=n)])
betas = np.array([1, -0.5, 0.5])
eta = X @ betas
lam = np.exp(eta)

y = np.random.poisson(lam)

# Ajusta com a função poi_reg
out_poi_reg = poi_reg(y, X)
beta_final = out_poi_reg[-1]

# Ajusta com statsmodels
model_glm = sm.GLM(y, X, family=sm.families.Poisson())
res_glm = model_glm.fit()

# Compara resultados
comparison = pd.DataFrame({
    "true": betas,
    "poi_reg": beta_final,
    "glm": res_glm.params
}, index=["Intercept", "x1", "x2"])

print(comparison)

           true   poi_reg       glm
Intercept   1.0  1.012195  1.012195
x1         -0.5 -0.529951 -0.529951
x2          0.5  0.406359  0.406359
